# Supplementary Results 12 — Ancestry and sample size as predictors of gene discovery

Does ancestry add anything to study size in explaining how many novel genes a GWAS finds? A
negative-binomial model of novel genes per study on the non-Finnish European proportion, effective
sample size and publication year, with standard errors clustered on cohort.

Numbers are written to `results/sr12_discovery_regression.json`. The two main-text incidence-rate
ratios of Results 1 are registered here too, since this is the only place they are computed.

**Provenance.** `chapters/_legacy/06-review-r1/ancestry-vs-sample-size/01_discovery_regression.ipynb`
— the same novelty definition, the same Cameron-Trivedi dispersion held fixed, the same clustering
and the same specifications.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from manuscript_methods import discovery, paper

numbers = {}
MAX_YEAR = discovery.MAX_YEAR

## The per-study modelling table

A gene counts as novel for a study when no study published earlier reported it; every study sharing
the earliest publication date is credited. Consortium releases cluster on the project, everything
else on the publication.

In [2]:
studies = pd.read_parquet(paper.derived("study_annotation"))
# `study_annotation` carries everything but the publication identifier, which the clustering needs.
publications = pd.read_parquet(paper.release("study"), columns=["studyId", "pubmedId"])
studies = studies.merge(publications, on="studyId", how="left")
studies["publicationDate"] = pd.to_datetime(studies["publicationDate"], errors="coerce")
studies["cluster"] = np.where(
    studies["projectId"].ne("GCST"), studies["projectId"], studies["pubmedId"].fillna(studies["studyId"])
)

# `study_annotation.effectiveSampleSize` is null for almost every quantitative study, so it is
# recomputed here as the pipeline defines it: binary traits use prevalence * (1 - prevalence) * N,
# quantitative traits use N. Reading the column instead leaves 99 measurement studies modelled.
n_samples = studies["nSamples"].astype("float64")
cases = studies["nCases"].astype("float64")
controls = studies["nControls"].astype("float64")
is_binary = cases.notna() & controls.notna() & (cases > 0) & (controls > 0)
prevalence = (cases / n_samples).where(is_binary)
studies["effectiveSampleSize"] = np.where(is_binary, prevalence * (1 - prevalence) * n_samples, n_samples)
print(f"GWAS studies: {len(studies):,} | clusters: {studies['cluster'].nunique():,}")
print(studies["ancestryClass"].value_counts().to_string())

GWAS studies: 100,526 | clusters: 7,005
ancestryClass
EUR        65253
non-EUR    23548
mixed      11725


In [3]:
def build_study_table(name, label):
    """One row per study: novel genes, all genes, credible sets, and the study covariates."""
    rows = pd.read_parquet(paper.derived(name), columns=["studyId", "studyLocusId", "geneId"])
    rows = rows.merge(studies[["studyId", "publicationDate", "year"]], on="studyId", how="inner")
    rows = rows[rows["year"].notna() & (rows["year"] <= MAX_YEAR)]

    first_date = rows.groupby("geneId")["publicationDate"].transform("min")
    rows["is_novel"] = rows["publicationDate"].eq(first_date)

    per_study = rows.groupby("studyId").agg(n_genes=("geneId", "nunique"), n_cs=("studyLocusId", "nunique"))
    novel = rows[rows["is_novel"]].groupby("studyId")["geneId"].nunique().rename("novel_genes")
    per_study = per_study.join(novel).fillna({"novel_genes": 0})
    per_study["novel_genes"] = per_study["novel_genes"].astype(int)
    return per_study.reset_index().merge(studies, on="studyId", how="inner").assign(domain=label)


def prepare(frame):
    """Drop studies the model cannot use and add the derived regressors."""
    out = frame[
        frame["effectiveSampleSize"].notna() & (frame["effectiveSampleSize"] > 0) & frame["year"].notna()
    ].copy()
    out["log10N"] = np.log10(out["effectiveSampleSize"])
    out["year_c"] = out["year"] - 2015
    out["ancestryClass"] = pd.Categorical(out["ancestryClass"], categories=["EUR", "non-EUR", "mixed"])
    return out


disease = prepare(build_study_table("prioritised_genes_diseases", "disease"))
measurement = prepare(build_study_table("prioritised_genes_measurements", "measurement"))
DOMAINS = [("disease", disease), ("measurement", measurement)]

numbers["S12.01"] = len(disease)
numbers["S12.02"] = disease["cluster"].nunique()
print(f"disease studies modelled: {len(disease):,} ({numbers['S12.02']:,} clusters)")
print(f"measurement studies modelled: {len(measurement):,} ({measurement['cluster'].nunique():,} clusters)")

disease studies modelled: 5,349 (1,525 clusters)
measurement studies modelled: 21,491 (1,604 clusters)


In [4]:
for label, frame in DOMAINS:
    mean, variance = frame["novel_genes"].mean(), frame["novel_genes"].var()
    print(f"{label:>12}: mean {mean:6.2f}  variance {variance:9.2f}  ratio {variance / mean:6.2f}")
numbers["S12.03"] = round(float(disease["novel_genes"].var() / disease["novel_genes"].mean()), 0)

     disease: mean   1.90  variance     40.58  ratio  21.32
 measurement: mean   1.15  variance     75.02  ratio  65.37


## Fitting

NB2 by direct maximum likelihood overflows on this data, so the dispersion is estimated once by the
Cameron-Trivedi auxiliary regression from a Poisson fit and then held fixed, and the model is a GLM
at that dispersion with cluster-robust standard errors.

In [5]:
FORMULAS = {
    "M1: EUR + N": "novel_genes ~ nfeFraction + log10N",
    "M2: + year": "novel_genes ~ nfeFraction + log10N + year_c",
    "M3: ancestry class": "novel_genes ~ C(ancestryClass, Treatment('EUR')) + log10N + year_c",
}


def estimate_alpha(frame, formula):
    """NB2 dispersion by the Cameron-Trivedi auxiliary regression on a Poisson fit."""
    poisson = smf.glm(formula, data=frame, family=sm.families.Poisson()).fit()
    mu = poisson.mu
    y = np.asarray(poisson.model.endog, dtype="float64")
    auxiliary = ((y - mu) ** 2 - y) / mu
    return max(float(sm.OLS(auxiliary, mu).fit().params[0]), 1e-6)


def fit_nb(frame, formula, alpha):
    """NB2 as a GLM at fixed dispersion, standard errors clustered on cohort."""
    model = smf.glm(formula, data=frame, family=sm.families.NegativeBinomial(alpha=alpha))
    return model.fit(cov_type="cluster", cov_kwds={"groups": frame["cluster"].astype("category").cat.codes})


ALPHA = {label: estimate_alpha(frame, FORMULAS["M2: + year"]) for label, frame in DOMAINS}
fits = {
    (label, name): fit_nb(frame, formula, ALPHA[label])
    for label, frame in DOMAINS
    for name, formula in FORMULAS.items()
}
print({k: round(v, 3) for k, v in ALPHA.items()})

{'disease': 3.718, 'measurement': 3.79}


In [6]:
def irr(result, term, invert=False):
    """Incidence-rate ratio and interval for one term, optionally in the reversed direction."""
    interval = result.conf_int()
    coefficient = float(result.params[term])
    low, high = float(interval.loc[term, 0]), float(interval.loc[term, 1])
    if invert:
        coefficient, low, high = -coefficient, -high, -low
    return {
        "IRR": float(np.exp(coefficient)),
        "ci_low": float(np.exp(low)),
        "ci_high": float(np.exp(high)),
        "P": float(result.pvalues[term]),
    }


primary = fits[("disease", "M2: + year")]
european = irr(primary, "nfeFraction", invert=True)
forward = irr(primary, "nfeFraction")
size = irr(primary, "log10N")
year = irr(primary, "year_c")

numbers["S12.04"] = round(european["IRR"], 2)
numbers["S12.05"] = round(european["ci_low"], 2)
numbers["S12.06"] = round(european["ci_high"], 2)
numbers["S12.07"] = float(european["P"])
numbers["S12.08"] = round(forward["IRR"], 2)
numbers["S12.09"] = round(size["IRR"], 2)
numbers["S12.10"] = round(size["ci_low"], 2)
numbers["S12.11"] = round(size["ci_high"], 2)
numbers["S12.12"] = float(size["P"])
numbers["S12.13"] = round(year["IRR"], 2)
numbers["S12.14"] = round(year["ci_low"], 2)
numbers["S12.15"] = round(year["ci_high"], 2)
numbers["S12.16"] = float(year["P"])
print(
    f"fully non-European vs fully European: IRR {numbers['S12.04']} "
    f"[{numbers['S12.05']}, {numbers['S12.06']}], P {numbers['S12.07']:.1e}"
)
print(f"tenfold larger study: IRR {numbers['S12.09']} | per year: IRR {numbers['S12.13']}")

fully non-European vs fully European: IRR 1.69 [1.3, 2.2], P 1.0e-04
tenfold larger study: IRR 3.67 | per year: IRR 0.84


In [7]:
# By ancestry class rather than the continuous proportion.
class_terms = {
    "non-EUR": "C(ancestryClass, Treatment('EUR'))[T.non-EUR]",
    "mixed": "C(ancestryClass, Treatment('EUR'))[T.mixed]",
}
class_table = pd.DataFrame(
    [
        {"domain": label, "class": name, **irr(fits[(label, "M3: ancestry class")], term)}
        for label, _ in DOMAINS
        for name, term in class_terms.items()
    ]
)
numbers["S12.17"] = round(float(class_table.iloc[0]["IRR"]), 2)
numbers["S12.18"] = round(float(class_table.iloc[0]["ci_low"]), 2)
numbers["S12.19"] = round(float(class_table.iloc[0]["ci_high"]), 2)
numbers["S12.20"] = float(class_table.iloc[0]["P"])
numbers["S12.21"] = round(float(class_table.iloc[1]["IRR"]), 2)
numbers["S12.22"] = round(float(class_table.iloc[1]["ci_low"]), 2)
numbers["S12.23"] = round(float(class_table.iloc[1]["ci_high"]), 2)
numbers["S12.24"] = round(float(class_table.iloc[1]["P"]), 2)
# Results 1 quotes the continuous-proportion estimate for non-EUR against EUR (1.69) and the
# class-model estimate for mixed against EUR (0.97).
numbers["R1.34"] = numbers["S12.04"]
numbers["R1.36"] = numbers["S12.09"]
numbers["R1.35"] = numbers["S12.21"]
class_table.round(4)

,domain,class,IRR,ci_low,ci_high,P
0,disease,non-EUR,1.6670,1.3016,2.1351,0.0001
1,disease,mixed,0.9731,0.8032,1.1789,0.7805
2,measurement,non-EUR,3.8441,1.9893,7.4280,0.0001
3,measurement,mixed,0.6248,0.3074,1.2699,0.1937


## Measurements, and how the ancestry term depends on the year specification

In [8]:
measurement_fit = fits[("measurement", "M2: + year")]
measurement_european = irr(measurement_fit, "nfeFraction", invert=True)
measurement_size = irr(measurement_fit, "log10N")
numbers["S12.25"] = round(measurement_european["IRR"], 2)
numbers["S12.26"] = round(float(class_table.iloc[2]["IRR"]), 2)
numbers["S12.27"] = round(float(class_table.iloc[2]["ci_low"]), 2)
numbers["S12.28"] = round(float(class_table.iloc[2]["ci_high"]), 2)
numbers["S12.29"] = round(float(class_table.iloc[3]["IRR"]), 2)
numbers["S12.30"] = round(float(class_table.iloc[3]["ci_low"]), 2)
numbers["S12.31"] = round(float(class_table.iloc[3]["ci_high"]), 2)
numbers["S12.32"] = round(float(class_table.iloc[3]["P"]), 2)
numbers["S12.33"] = round(measurement_size["IRR"], 2)
numbers["S12.34"] = round(measurement_size["ci_low"], 2)
numbers["S12.35"] = round(measurement_size["ci_high"], 2)
print({k: numbers[k] for k in ["S12.25", "S12.26", "S12.29", "S12.33"]})

{'S12.25': 3.37, 'S12.26': 3.84, 'S12.29': 0.62, 'S12.33': 5.95}


In [9]:
specifications = []
for label, frame in DOMAINS:
    for name, formula in [
        ("year linear", "novel_genes ~ nfeFraction + log10N + year_c"),
        ("year factor", "novel_genes ~ nfeFraction + log10N + C(year)"),
        ("no year", "novel_genes ~ nfeFraction + log10N"),
    ]:
        result = fit_nb(frame, formula, estimate_alpha(frame, formula))
        specifications.append({"domain": label, "spec": name, **irr(result, "nfeFraction", invert=True)})
specification_table = pd.DataFrame(specifications)

numbers["S12.36"] = round(float(specification_table.iloc[1]["IRR"]), 2)
numbers["S12.37"] = round(float(specification_table.iloc[2]["IRR"]), 2)
numbers["S12.38"] = round(float(specification_table.iloc[2]["P"]), 2)
specification_table.round(4)

,domain,spec,IRR,ci_low,ci_high,P
0,disease,year linear,1.6877,1.2962,2.1976,0.0001
1,disease,year factor,2.0094,1.4310,2.8216,0.0001
2,disease,no year,0.8991,0.7381,1.0953,0.2909
3,measurement,year linear,3.3686,1.8459,6.1472,0.0001
4,measurement,year factor,2.6250,1.6377,4.2076,0.0001
5,measurement,no year,1.8475,0.8337,4.0937,0.1305


In [10]:
# European share of studies before and after 2018, which is why year has to be in the model.
published = studies[studies["year"].notna() & (studies["year"] <= MAX_YEAR)]
early = published[published["year"] <= 2017]
late = published[published["year"] >= 2018]
numbers["S12.39"] = round(100 * float((early["ancestryClass"] == "EUR").mean()), 1)
numbers["S12.40"] = round(100 * float((late["ancestryClass"] == "EUR").mean()), 1)
print(f"European studies: {numbers['S12.39']}% up to 2017, {numbers['S12.40']}% from 2018")

European studies: 76.9% up to 2017, 64.5% from 2018


## Sensitivity — other outcomes, sample-size quintiles, and leaving cohorts out

In [11]:
sensitivity = []
for label, frame in DOMAINS:
    for outcome in ["novel_genes", "n_genes", "n_cs"]:
        formula = f"{outcome} ~ nfeFraction + log10N + year_c"
        result = fit_nb(frame, formula, estimate_alpha(frame, formula))
        sensitivity.append({"domain": label, "outcome": outcome, **irr(result, "nfeFraction", invert=True)})
sensitivity_table = pd.DataFrame(sensitivity)
numbers["S12.41"] = round(float(sensitivity_table.iloc[1]["IRR"]), 2)
numbers["S12.42"] = round(float(sensitivity_table.iloc[2]["IRR"]), 2)
sensitivity_table.round(4)

,domain,outcome,IRR,ci_low,ci_high,P
0,disease,novel_genes,1.6877,1.2962,2.1976,0.0001
1,disease,n_genes,1.5096,1.2075,1.8873,0.0003
2,disease,n_cs,1.5463,1.2660,1.8887,0.0000
3,measurement,novel_genes,3.3686,1.8459,6.1472,0.0001
4,measurement,n_genes,1.9454,1.0336,3.6615,0.0392
5,measurement,n_cs,2.0330,1.1006,3.7553,0.0234


In [12]:
quintiles = disease.copy()
quintiles["quintile"] = pd.qcut(quintiles["log10N"], 5, labels=[f"Q{i}" for i in range(1, 6)])
quintiles["band"] = pd.cut(
    quintiles["nfeFraction"], [-0.01, 0.1, 0.9, 1.01], labels=["<10% EUR", "10-90% EUR", ">90% EUR"]
)
means = quintiles.pivot_table(index="quintile", columns="band", values="novel_genes", aggfunc="mean", observed=False)
numbers["S12.43"] = round(float(means.loc["Q5", "<10% EUR"]), 2)
numbers["S12.44"] = round(float(means.loc["Q5", ">90% EUR"]), 2)
means.round(2)

band,<10% EUR,10-90% EUR,>90% EUR
quintile,,,
Q1,0.95,0.60,1.04
Q2,0.60,0.55,1.12
Q3,0.84,0.66,1.58
Q4,1.64,1.24,2.17
Q5,8.07,3.79,4.42


In [13]:
def leave_one_out(frame, top=10):
    """Refit the primary model dropping each of the largest cohorts in turn."""
    formula = FORMULAS["M2: + year"]
    rows = []
    for excluded in [None, *frame["cluster"].value_counts().head(top).index]:
        subset = frame if excluded is None else frame[frame["cluster"] != excluded]
        result = fit_nb(subset, formula, estimate_alpha(subset, formula))
        rows.append(
            {
                "excluded": "(none)" if excluded is None else excluded,
                "studies": len(subset),
                **irr(result, "nfeFraction", invert=True),
            }
        )
    return pd.DataFrame(rows)


loco = leave_one_out(disease)
dropped = loco[loco["excluded"] != "(none)"]
numbers["S12.45"] = round(float(dropped["IRR"].min()), 2)
numbers["S12.46"] = round(float(dropped["IRR"].max()), 2)

finngen = dropped[dropped["excluded"].astype(str).str.contains("FINNGEN", case=False)]
if len(finngen):
    numbers["S12.47"] = round(float(finngen.iloc[0]["IRR"]), 2)
    numbers["S12.48"] = round(float(finngen.iloc[0]["ci_low"]), 2)
    numbers["S12.49"] = round(float(finngen.iloc[0]["ci_high"]), 2)
    numbers["S12.50"] = round(float(finngen.iloc[0]["P"]), 3)
print(f"IRR across leave-one-cohort-out refits: {numbers['S12.45']} to {numbers['S12.46']}")
loco.round(4)

IRR across leave-one-cohort-out refits: 1.46 to 1.72


,excluded,studies,IRR,ci_low,ci_high,P
0,(none),5349,1.6877,1.2962,2.1976,0.0001
1,39024449,3911,1.4772,1.1491,1.8990,0.0023
2,FINNGEN_R12,4509,1.4590,0.9926,2.1444,0.0546
3,34737426,5156,1.6198,1.2235,2.1444,0.0008
4,30104761,5165,1.6564,1.2585,2.1801,0.0003
5,34594039,5218,1.6648,1.2675,2.1867,0.0002
6,36777996,5296,1.7003,1.3054,2.2147,0.0001
7,33959723,5301,1.6767,1.2820,2.1930,0.0002
8,34662886,5310,1.6684,1.2780,2.1782,0.0002
9,32514122,5318,1.7178,1.3328,2.2141,0.0000


## Write the results

In [14]:
print(paper.save_results("sr12_discovery_regression", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr12_discovery_regression.json


,computed
S12.01,5.349000e+03
S12.02,1.525000e+03
S12.03,2.100000e+01
S12.04,1.690000e+00
S12.05,1.300000e+00
S12.06,2.200000e+00
S12.07,1.017779e-04
S12.08,5.900000e-01
S12.09,3.670000e+00
S12.10,2.750000e+00
